[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/15_django/lab01_django_in_a_notebook.ipynb)

# 🧪 Lab 1 — Django in a notebook: ORM, templates, views & forms

> **Module:** Django for AI Web Apps (Module 15) · **Estimated time:** ~60 minutes · **Difficulty:** Intermediate

Module 15 is a *mini-book*: you read [why-django.md](why-django.md), [project-anatomy.md](project-anatomy.md), [models-and-admin.md](models-and-admin.md) and [views-templates-forms.md](views-templates-forms.md), and you run the **ChurnScope** example app from a terminal. This lab is the hands-on companion to those four chapters — and it runs **real Django inside this notebook**. No `manage.py`, no dev server, no browser: we configure settings by hand, create a real SQLite database, define the *same* `Prediction` model ChurnScope uses, render real templates, and hit real views through Django's test client — all in this kernel.

Why bother, when the example app is one `runserver` away? Because a running web server is a black box: you POST a form and HTML comes back. In a notebook you can hold each piece in your hands — *this* is the ORM writing SQL, *this* is the template engine filling holes, *this* is a view being called with a request. You've already served models with **FastAPI** (Modules 7 and 12) — a thin API layer; Django is the *batteries-included* counterpart, and this lab unpacks the batteries one by one. When you then open `example-app/`, every file will look familiar.

> 🧭 **Mental model: the request pipeline.** Everything Django does is one assembly line —
> **request → URLconf → view → model/ORM → template → response.**
> A request arrives; the URLconf decides *which view* handles it; the view talks to the database
> through the *model*, hands the data to a *template*, and returns the filled-in HTML (or JSON) as
> the response. That's the **MTV** cycle from [why-django.md](why-django.md), in motion —
> this lab walks the line station by station, then sends one request through the whole pipeline.

If you've read [project-anatomy.md](project-anatomy.md), here's the honest framing: this notebook is one long `python manage.py shell` session — except we also *build the wiring* (settings, the app package, the URLconf) ourselves, so you see exactly what `startproject` normally hides.

## ✅ Prerequisites

NB 6 (classes — models are classes) and NB 13 (SQL — you'll recognise what the ORM writes) help a lot; NB 27 built the churn-scoring POC that ChurnScope wraps; Module 11 (NB 39–40) gives the production framing. Django is the only extra dependency — §1 checks for it. **This is an optional module**: skip or skim guilt-free.

## 🎯 Learning objectives

By the end of this lab you can:

1. Boot Django **without `manage.py`** — `settings.configure()` + `django.setup()` — and explain what `startproject` normally generates for you.
2. Define a **model**, get its table created, and explain what **migrations** add on top.
3. Query with the **ORM** — `filter`, `exclude`, `order_by`, aggregates — and read the SQL a lazy QuerySet compiles to.
4. Render **templates** with variables, loops and filters — from strings and through a loader.
5. Route URLs to **views** and test the full request → response cycle with Django's `Client` — no server, no browser.
6. Validate untrusted input with a **form**, and POST it through the same pipeline the ChurnScope app uses.

## 1. Smoke test — is Django installed?

The course venv ships Django (elsewhere: `pip install django`, or uncomment the `%pip` line below on Colab). Like the optional-library appendices (the PyTorch track, for example), this lab degrades gracefully: every Django cell checks `HAS_DJANGO` and prints a skip note instead of crashing, so the notebook stays readable in an environment without Django.

In [1]:
# On Colab or a fresh machine: uncomment the next line, run it, then restart the kernel.
# %pip install django

try:
    import django
    HAS_DJANGO = True
    print(f"Django {django.get_version()} — ready.")
except ImportError:
    HAS_DJANGO = False
    print("Django not installed — install with:  pip install django")
    print("Every Django cell below will print a skip note instead of running.")

Django 5.2.15 — ready.


## 2. A whole project in one call — `settings.configure()`

`django-admin startproject churnscope` writes a handful of files whose real job is to answer one question: **what are the settings?** Which apps are installed, where is the database, how are templates found, which module maps URLs. In a script or a notebook you can answer it directly: call `settings.configure(...)` once, then `django.setup()`.

Two honest adaptations for notebook life, both straight from [project-anatomy.md](project-anatomy.md)'s vocabulary:

- **An app is just a Python package.** ChurnScope's `scoring/` app is a folder with `__init__.py`, `models.py`, `views.py`, … For `INSTALLED_APPS` to accept `"scoring"`, the package must be importable — so we write a *two-file* `scoring` package into a temp folder and put that folder on `sys.path`. (We leave its `models.py` as a stub and define the model in a notebook cell instead, so you can watch it happen — §3.)
- **Templates come from a loader.** The real app loads template *files* from `scoring/templates/` (`APP_DIRS=True`); we swap in Django's `locmem` loader — a plain dict mapping template names to template strings — so templates can live in notebook cells. Same engine, same syntax, different storage.

| Setting | ChurnScope (`churnscope/settings.py`) | This notebook |
|---|---|---|
| `INSTALLED_APPS` | admin, auth, contenttypes, sessions, … + `scoring` | contenttypes, auth + `scoring` |
| `DATABASES` | SQLite file next to `manage.py` | SQLite file in the temp dir (fresh per kernel) |
| `TEMPLATES` | files, via `APP_DIRS=True` | strings, via the `locmem` loader |
| `ROOT_URLCONF` | `"churnscope.urls"` (a module of URL patterns) | `"notebook_urls"` (a module we build in §6) |
| `DEBUG` / `ALLOWED_HOSTS` | env-driven (see the deployment chapter) | `True` / `["testserver"]` (the test client's hostname) |

In [2]:
import os
import sys
import tempfile
from pathlib import Path

if HAS_DJANGO:
    from django.conf import settings

    # Jupyter runs an asyncio event loop; Django's ORM refuses to run in async
    # contexts unless told otherwise — 🔬 deep dive right below.
    os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

    LAB_DIR = Path(tempfile.gettempdir()) / "churnscope_lab"
    DB_PATH = LAB_DIR / "churnscope_lab.sqlite3"
    # One dict = the whole template store. Keep the same object across re-runs:
    TEMPLATE_REGISTRY = globals().get("TEMPLATE_REGISTRY", {})

    if settings.configured:
        print("Settings already configured — restart the kernel to reconfigure.")
    else:
        # --- the 'scoring' app: an app is just an importable package --------
        (LAB_DIR / "scoring").mkdir(parents=True, exist_ok=True)
        (LAB_DIR / "scoring" / "__init__.py").write_text(
            "# The scoring app — same name as ChurnScope's app package.\n")
        (LAB_DIR / "scoring" / "models.py").write_text(
            "# In the example app, the Prediction model lives in this file.\n"
            "# In this lab we define it in a notebook cell instead (lab §3).\n")
        sys.path.insert(0, str(LAB_DIR))

        if DB_PATH.exists():              # fresh database on every kernel restart
            DB_PATH.unlink()

        settings.configure(
            DEBUG=True,                                # keeps a query log (§4)
            SECRET_KEY="lab-only-not-a-real-secret",
            ALLOWED_HOSTS=["testserver"],              # the test client's host
            INSTALLED_APPS=[
                "django.contrib.contenttypes",
                "django.contrib.auth",
                "scoring",                             # ← our app
            ],
            DATABASES={"default": {
                "ENGINE": "django.db.backends.sqlite3",
                "NAME": DB_PATH,
            }},
            ROOT_URLCONF="notebook_urls",              # the module we build in §6
            TEMPLATES=[{
                "BACKEND": "django.template.backends.django.DjangoTemplates",
                "OPTIONS": {"loaders": [
                    ("django.template.loaders.locmem.Loader", TEMPLATE_REGISTRY),
                ]},
            }],
            USE_TZ=True,
            TIME_ZONE="UTC",
            DEFAULT_AUTO_FIELD="django.db.models.BigAutoField",
        )
        django.setup()                     # populate the app registry — once!

    from django.apps import apps

    print("Installed apps:", [a.label for a in apps.get_app_configs()])
    print("Database file :", settings.DATABASES["default"]["NAME"])
else:
    print("Django not installed — skipping.")

Installed apps: ['contenttypes', 'auth', 'scoring']
Database file : /var/folders/sz/1k1y5gg975j3mc23vxwrt0v40000gn/T/churnscope_lab/churnscope_lab.sqlite3


### 🔬 What's `DJANGO_ALLOW_ASYNC_UNSAFE` about?

Jupyter's kernel runs your cells inside an **asyncio event loop**. Django notices: since 3.x, every ORM call checks "am I inside a running event loop?" and raises `SynchronousOnlyOperation` if so — because a *blocking* database call inside an async web server would freeze every other request it's juggling. That guard is exactly right in production and exactly wrong in a notebook, where blocking is fine. So Django ships the escape hatch above (tools like `django-extensions`' `shell_plus --notebook` set it for you). Curious? Comment the line out, restart the kernel, re-run — and meet the error with your own eyes.

> ⚠️ **Pitfall: settings are write-once.** `settings.configure()` may only run once per process, and most values are read at `django.setup()` time — you cannot hot-swap `INSTALLED_APPS` afterwards. That's why the cell above guards on `settings.configured` (re-running is a harmless no-op) and why *changing* the configuration means **restarting the kernel**. The real project has the same rule in different clothes: edit `settings.py` → restart the server.

## 3. The model — `Prediction`, straight from ChurnScope

[models-and-admin.md](models-and-admin.md) in one line: *a model is a Python class; each attribute is a column.* Below is ChurnScope's `Prediction` model, field-for-field from `example-app/scoring/models.py`, plus **one** notebook-only line: `app_label = "scoring"`. In the real project the file's *location* (inside the `scoring` package) tells Django which app owns the model; a notebook cell has no location, so we pin ownership explicitly.

Business meaning first: every time ChurnScope scores a customer it logs *what went in* (tenure, charges, tickets, contract), *what came out* (probability, verdict), and *when* — an **audit trail** you can analyse later ("are month-to-month customers really riskier?") or show to a regulator asking why the model flagged someone.

> ♻️ Re-running the class cell in a live session prints a `RuntimeWarning: Model … was already registered` — Django telling you it kept the registration. Harmless here.

In [3]:
if HAS_DJANGO:
    from django.db import models

    class Prediction(models.Model):
        """One churn score, logged for the admin dashboard and audit trail."""

        created_at = models.DateTimeField(auto_now_add=True)   # set once, on INSERT
        tenure_months = models.PositiveIntegerField()
        monthly_charges = models.FloatField()
        support_tickets = models.PositiveIntegerField()
        contract = models.CharField(max_length=20)
        probability = models.FloatField()
        will_churn = models.BooleanField()

        class Meta:
            app_label = "scoring"        # notebook-only: pin the model to our app
            ordering = ["-created_at"]   # newest first — everywhere, by default

        def __str__(self):
            verdict = "churn" if self.will_churn else "stay"
            return f"{self.contract}: p={self.probability:.2f} ({verdict})"

    print("Models registered to 'scoring':",
          [m.__name__ for m in apps.get_app_config("scoring").get_models()])
    print("Columns:", [f.name for f in Prediction._meta.fields])
else:
    print("Django not installed — skipping.")

Models registered to 'scoring': ['Prediction']
Columns: ['id', 'created_at', 'tenure_months', 'monthly_charges', 'support_tickets', 'contract', 'probability', 'will_churn']


### 🔬 You never write `CREATE TABLE`

The class *is* the schema. You can ask Django's schema editor for the SQL it would run — `collect_sql=True` collects instead of executing. NB 13 readers: notice the `id` primary key you never declared, and the `CHECK` constraints derived from `PositiveIntegerField`.

In [4]:
if HAS_DJANGO:
    from django.db import connection

    with connection.schema_editor(collect_sql=True) as editor:
        editor.create_model(Prediction)

    for statement in editor.collected_sql:
        print(statement)
else:
    print("Django not installed — skipping.")

CREATE TABLE "scoring_prediction" ("id" integer NOT NULL PRIMARY KEY AUTOINCREMENT, "created_at" datetime NOT NULL, "tenure_months" integer unsigned NOT NULL CHECK ("tenure_months" >= 0), "monthly_charges" real NOT NULL, "support_tickets" integer unsigned NOT NULL CHECK ("support_tickets" >= 0), "contract" varchar(20) NOT NULL, "probability" real NOT NULL, "will_churn" bool NOT NULL);


### Migrations — Git commits for your schema

In the real project you'd now run two `manage.py` commands: **`makemigrations`** (diff the models against the last recorded schema and write a versioned migration file — ChurnScope's first is `scoring/migrations/0001_initial.py`) and **`migrate`** (apply the pending ones). Migrations are reviewable, revertible, and keep every environment in sync — [models-and-admin.md](models-and-admin.md) calls them *Git commits for the database*, which is exactly the right mental file.

There's no `manage.py` here, but every management command is also a plain function: `call_command("migrate", ...)`. Our notebook app has no migration files, so we pass `run_syncdb=True` — "for apps without migrations, just create their tables directly". (That's also why §2 wrote a stub `models.py`: `run_syncdb` only considers apps that *have* a models module.) The `auth` and `contenttypes` apps ship real migrations, so those get applied properly in the same call.

In [5]:
if HAS_DJANGO:
    from django.core.management import call_command

    call_command("migrate", run_syncdb=True, verbosity=0)   # ≙ python manage.py migrate

    print("Tables now in the database:")
    for name in sorted(connection.introspection.table_names()):
        print(f"  {name}{'   ← ours' if name == 'scoring_prediction' else ''}")
else:
    print("Django not installed — skipping.")

Tables now in the database:
  auth_group
  auth_group_permissions
  auth_permission
  auth_user
  auth_user_groups
  auth_user_user_permissions
  django_content_type
  django_migrations
  scoring_prediction   ← ours


## 4. The ORM — create, query, aggregate

We need something to log. Meet the **scorer** — copied from `example-app/scoring/scorer.py`, itself a transparent stand-in for the churn model you built in the Module 7 POC (NB 27). Note what it *isn't*: there's no Django in it. It's a plain function — exactly the seam that later lets you swap in a pickled scikit-learn model without touching views, forms or templates (that swap is [serving-a-model.md](serving-a-model.md)'s story, and Lab 2's).

In [6]:
import math

CONTRACT_RISK = {"month-to-month": 0.9, "one-year": 0.0, "two-year": -0.9}


def churn_probability(*, tenure_months, monthly_charges, support_tickets, contract):
    """Return P(churn) in [0, 1] for one customer — same logic as the example app."""
    z = (
        -0.5
        + 1.4 * CONTRACT_RISK.get(contract, 0.0)
        - 0.05 * float(tenure_months)
        + 0.015 * float(monthly_charges)
        + 0.25 * float(support_tickets)
    )
    return 1.0 / (1.0 + math.exp(-z))


demo = churn_probability(tenure_months=2, monthly_charges=95,
                         support_tickets=5, contract="month-to-month")
print(f"P(churn) for a fresh, ticket-heavy month-to-month customer: {demo:.3f}")

P(churn) for a fresh, ticket-heavy month-to-month customer: 0.966


Now score a small book of customers and **log every prediction** — one ORM call per row, no SQL in sight. We use `get_or_create` instead of `create`: it looks for a row with these exact field values first and only inserts when none exists. That makes the cell **idempotent** — safe to re-run, the same property NB 40 demanded of every scheduled task. (`create()` always inserts: run a `create()` cell five times, get five rows.)

In [7]:
if HAS_DJANGO:
    SAMPLE_CUSTOMERS = [
        dict(tenure_months=2,  monthly_charges=95.0, support_tickets=5, contract="month-to-month"),
        dict(tenure_months=7,  monthly_charges=82.5, support_tickets=3, contract="month-to-month"),
        dict(tenure_months=13, monthly_charges=99.0, support_tickets=1, contract="month-to-month"),
        dict(tenure_months=18, monthly_charges=64.0, support_tickets=2, contract="one-year"),
        dict(tenure_months=32, monthly_charges=71.0, support_tickets=0, contract="one-year"),
        dict(tenure_months=55, monthly_charges=58.5, support_tickets=1, contract="two-year"),
    ]

    for customer in SAMPLE_CUSTOMERS:
        p = churn_probability(**customer)
        row, created = Prediction.objects.get_or_create(
            **customer,
            defaults={"probability": round(p, 4), "will_churn": p >= 0.5},
        )
        print("new:     " if created else "existing:", row)

    print("\nRows in scoring_prediction:", Prediction.objects.count())
else:
    print("Django not installed — skipping.")

new:      month-to-month: p=0.97 (churn)
new:      month-to-month: p=0.92 (churn)
new:      month-to-month: p=0.86 (churn)
new:      one-year: p=0.52 (churn)
new:      one-year: p=0.26 (stay)
new:      two-year: p=0.03 (stay)

Rows in scoring_prediction: 6


> 🧠 **Mental model: class = table, instance = row.** `Prediction` (the class) is the `scoring_prediction` table; each instance is one record; each attribute is one column; and `Prediction.objects` — the **manager** — is the gateway every query goes through. If you think in pandas: `objects.filter(...)` is to a database what boolean masking is to a DataFrame — except the work happens *in SQL, inside the database*, before Python ever sees a byte.

Reading rows back:

In [8]:
if HAS_DJANGO:
    oldest = Prediction.objects.order_by("pk").first()    # the first row we inserted
    print("type       :", type(oldest).__name__)
    print("__str__    :", oldest)                          # the model's own repr
    print("attributes :", oldest.pk, "|", oldest.tenure_months, "months |",
          oldest.contract, "|", oldest.probability)

    same_row = Prediction.objects.get(pk=oldest.pk)        # get() = exactly one row
    print("get(pk=…)  :", same_row)
else:
    print("Django not installed — skipping.")

type       : Prediction
__str__    : month-to-month: p=0.97 (churn)
attributes : 1 | 2 months | month-to-month | 0.9656
get(pk=…)  : month-to-month: p=0.97 (churn)


In [9]:
if HAS_DJANGO:
    print("churners       :", Prediction.objects.filter(will_churn=True).count())
    print("month-to-month :", Prediction.objects.filter(contract="month-to-month").count())
    print("everything else:", Prediction.objects.exclude(contract="month-to-month").count())

    # Field lookups: <field>__<lookup> — gt / gte / lt / lte / in / contains / …
    print("\nHigh risk (p > 0.8), riskiest first:")
    for row in Prediction.objects.filter(probability__gt=0.8).order_by("-probability"):
        print("  ", row)
else:
    print("Django not installed — skipping.")

churners       : 4
month-to-month : 3
everything else: 3

High risk (p > 0.8), riskiest first:
   month-to-month: p=0.97 (churn)
   month-to-month: p=0.92 (churn)
   month-to-month: p=0.86 (churn)


---

### ✋ Quick exercise (~2 min) — Log a brand-new customer

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

A new customer signs up: **1 month** tenure, **105.0** monthly charges, **6** support tickets, **month-to-month** contract. Score them with `churn_probability`, log the prediction with `get_or_create` (probability rounded to 4 decimals, `will_churn` at the 0.5 threshold), and print the row plus the new total count.

In [10]:
# ✍️ Your turn 👇
new_customer = dict(tenure_months=1, monthly_charges=105.0,
                    support_tickets=6, contract="month-to-month")

# p_new = churn_probability(...)
p_new = ...

# row, created = Prediction.objects.get_or_create(..., defaults={...})
# print(row, "| created this run:", created)
# print("total predictions logged:", Prediction.objects.count())

<details>
<summary>✅ <b>Solution</b></summary>

```python
if HAS_DJANGO:
    new_customer = dict(tenure_months=1, monthly_charges=105.0,
                        support_tickets=6, contract="month-to-month")
    p_new = churn_probability(**new_customer)
    row, created = Prediction.objects.get_or_create(
        **new_customer,
        defaults={"probability": round(p_new, 4), "will_churn": p_new >= 0.5},
    )
    print(row, "| created this run:", created)
    print("total predictions logged:", Prediction.objects.count())
else:
    print("Django not installed — skipping.")
```

`get_or_create` splits its arguments in two: the keyword arguments are the *lookup* ("is this row already there?"); `defaults` only applies when inserting. Because the lookup covers all four input fields, re-running finds the existing row (`created: False`) instead of duplicating it — an idempotent write, NB 40 style. Short tenure + many tickets + month-to-month is the textbook churn profile, so expect p ≈ 0.98.
</details>

### 🔬 QuerySets are lazy — and you can read their SQL

`Prediction.objects.filter(...)` runs **no SQL**. It builds a *description* of a query — a `QuerySet` — and only executes when you consume it (iterate, `list()`, `len()`, slice…). Two consequences: chaining `.filter().exclude().order_by()` costs nothing until the end, and you can inspect `str(qs.query)` to preview the SQL it *would* run. And because §2 set `DEBUG=True`, Django logs every query it *actually* executes to `connection.queries` — our lie detector for the whole section:

In [11]:
if HAS_DJANGO:
    from django.db import reset_queries

    reset_queries()                                        # clear the query log
    qs = Prediction.objects.filter(contract="month-to-month").order_by("-probability")
    print("queries executed after building qs :", len(connection.queries))
    print("SQL it WOULD run:\n   ", qs.query)

    rows = list(qs)                                        # ← NOW it hits the database
    print("\nqueries executed after list(qs)    :", len(connection.queries))
    print("rows fetched:", len(rows))
else:
    print("Django not installed — skipping.")

queries executed after building qs : 0
SQL it WOULD run:
    SELECT "scoring_prediction"."id", "scoring_prediction"."created_at", "scoring_prediction"."tenure_months", "scoring_prediction"."monthly_charges", "scoring_prediction"."support_tickets", "scoring_prediction"."contract", "scoring_prediction"."probability", "scoring_prediction"."will_churn" FROM "scoring_prediction" WHERE "scoring_prediction"."contract" = month-to-month ORDER BY "scoring_prediction"."probability" DESC

queries executed after list(qs)    : 1
rows fetched: 3


Look at that SQL: a parameter-bound `SELECT … WHERE … ORDER BY` you never wrote — injection-safe by construction (NB 13's parameterised-query rule, applied for you). For analysis work, `values()` yields plain dicts that drop straight into pandas, and `aggregate()` pushes arithmetic into the database itself:

In [12]:
if HAS_DJANGO:
    from django.db.models import Avg, Count, Max

    print(Prediction.objects.aggregate(
        n=Count("id"), avg_p=Avg("probability"), worst=Max("probability")))

    import pandas as pd

    df = pd.DataFrame(Prediction.objects.values(
        "contract", "tenure_months", "support_tickets", "probability", "will_churn"))
    display(df)
else:
    print("Django not installed — skipping.")

{'n': 6, 'avg_p': 0.5926, 'worst': 0.9656}


,contract,tenure_months,support_tickets,probability,will_churn
0,two-year,55,1,0.0328,False
1,one-year,32,0,0.2621,False
2,one-year,18,2,0.5150,True
3,month-to-month,13,1,0.8635,True
4,month-to-month,7,3,0.9166,True
5,month-to-month,2,5,0.9656,True


> ⚠️ **Pitfall: the N+1 query problem.** The ORM makes database access *feel* free, so the classic mistake is looping in Python where SQL could make one pass — for example one `COUNT` query per contract type. Three contract types → 3 round-trips is quaint; the same pattern over 10,000 customers (usually by touching a foreign key inside a loop) is 10,001 queries and a page that takes seconds. The fix is always the same: *push the loop into the database* — here with `values(...).annotate(...)` (a GROUP BY); in foreign-key code with `select_related` / `prefetch_related`. `connection.queries` catches it red-handed:

In [13]:
if HAS_DJANGO:
    CONTRACTS = ["month-to-month", "one-year", "two-year"]

    reset_queries()
    per_contract = {c: Prediction.objects.filter(contract=c).count() for c in CONTRACTS}
    print(f"loop in Python : {per_contract}   → {len(connection.queries)} queries")

    reset_queries()
    grouped = {r["contract"]: r["n"]
               for r in Prediction.objects.values("contract").annotate(n=Count("id"))}
    print(f"GROUP BY in SQL: {grouped}   → {len(connection.queries)} query")
else:
    print("Django not installed — skipping.")

loop in Python : {'month-to-month': 3, 'one-year': 2, 'two-year': 1}   → 3 queries
GROUP BY in SQL: {'month-to-month': 3, 'one-year': 2, 'two-year': 1}   → 1 query


The rest of CRUD: mutate an instance and `save()` it (one `UPDATE` of that row), update a whole queryset in a single SQL `UPDATE` without fetching anything, and `delete()` — which returns a tally of what it removed. We do all three to a scratch row, so the cell cleans up after itself and stays re-runnable:

In [14]:
if HAS_DJANGO:
    scratch = Prediction.objects.create(        # create() = plain INSERT, every time
        tenure_months=99, monthly_charges=1.0, support_tickets=0,
        contract="two-year", probability=0.01, will_churn=False)
    print("created:", scratch.pk, "|", scratch)

    scratch.support_tickets = 3                 # 1) instance → mutate → save()
    scratch.save()
    print("after save():", Prediction.objects.get(pk=scratch.pk).support_tickets, "tickets")

    touched = Prediction.objects.filter(pk=scratch.pk).update(probability=0.02)
    print("queryset .update() touched", touched, "row(s)")   # 2) one SQL UPDATE

    deleted_total, per_model = scratch.delete()              # 3) DELETE, with receipts
    print("deleted:", deleted_total, per_model)
    print("row count back to:", Prediction.objects.count())
else:
    print("Django not installed — skipping.")

created: 7 | two-year: p=0.01 (stay)
after save(): 3 tickets
queryset .update() touched 1 row(s)
deleted: 1 {'scoring.Prediction': 1}
row count back to: 6


## 5. Templates — HTML with holes

[views-templates-forms.md](views-templates-forms.md) again: templates are HTML with `{{ variables }}` and `{% tags %}`, plus **filters** that pipe a value through a formatter — `{{ p|floatformat:2 }}`, chained with `|` like a Unix pipe. In ChurnScope they're files under `scoring/templates/scoring/`; our §2 settings swapped the file loader for an in-memory dict, so here a template is just a string. Same engine — `engines["django"]` below is literally the object `render()` will use in §6.

In [15]:
if HAS_DJANGO:
    from django.template import engines

    django_engine = engines["django"]      # the engine our TEMPLATES setting built

    t = django_engine.from_string("{{ name }} — churn risk {{ p|floatformat:2 }}")
    print(t.render({"name": "ACME Corp     ", "p": 0.8123456}))
    print(t.render({"name": "Backwater GmbH", "p": 0.07}))
else:
    print("Django not installed — skipping.")

ACME Corp      — churn risk 0.81
Backwater GmbH — churn risk 0.07


Templates deliberately allow only *display* logic — loops, conditionals, filters. Anything smarter belongs in the view. The `{% empty %}` clause and the `forloop.counter` variable below are the bread and butter of every results table; note that a QuerySet drops straight into the context, and the template pulls attributes off each model instance:

In [16]:
if HAS_DJANGO:
    report_tpl = django_engine.from_string(
        "CHURN WATCHLIST\n"
        "{% for p in predictions %}"
        "{{ forloop.counter }}. {{ p.contract|ljust:16 }} p={{ p.probability|floatformat:2 }}"
        " → {% if p.will_churn %}CALL THEM{% else %}relax{% endif %}\n"
        "{% empty %}No predictions logged yet.\n"
        "{% endfor %}")

    print(report_tpl.render({"predictions": Prediction.objects.order_by("-probability")[:4]}))
else:
    print("Django not installed — skipping.")

CHURN WATCHLIST
1. month-to-month   p=0.97 → CALL THEM
2. month-to-month   p=0.92 → CALL THEM
3. month-to-month   p=0.86 → CALL THEM
4. one-year         p=0.52 → CALL THEM



---

### ✋ Quick exercise (~2 min) — A risk-badge template

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Write a template string that renders one prediction `p` as a badge: the contract, then **`HIGH RISK`** if `p.probability >= 0.8`, **`medium`** if it's at least 0.5, else **`low`** — followed by the probability to 1 decimal in parentheses. (`{% if %}` / `{% elif %}` / `{% else %}` work just like Python's.) Render it for the three riskiest predictions.

In [17]:
# ✍️ Your turn 👇
# badge_tpl = django_engine.from_string("{{ p.contract }}: {% if ... %} ... {% endif %} ...")
badge_tpl = ...

# for pred in Prediction.objects.order_by("-probability")[:3]:
#     print(badge_tpl.render({"p": pred}))

<details>
<summary>✅ <b>Solution</b></summary>

```python
if HAS_DJANGO:
    badge_tpl = django_engine.from_string(
        "{{ p.contract }}: "
        "{% if p.probability >= 0.8 %}HIGH RISK"
        "{% elif p.probability >= 0.5 %}medium"
        "{% else %}low{% endif %}"
        " ({{ p.probability|floatformat:1 }})")

    for pred in Prediction.objects.order_by("-probability")[:3]:
        print(badge_tpl.render({"p": pred}))
else:
    print("Django not installed — skipping.")
```

Comparisons (`>=`) are allowed inside `{% if %}`, and the branches read exactly like Python. The cut here is deliberate, though: *thresholds* are display logic at best — if `0.8` means something to the business, compute the label in the view (or model) and hand the template a finished string. Templates should stay too dumb to be wrong.
</details>

## 6. URLs → views — the request pipeline, live

A **view** is a function: request in, response out. The **URLconf** decides which view gets which path — in ChurnScope, `churnscope/urls.py` includes `scoring/urls.py`, which maps `""` → `views.index`. Our §2 settings promised `ROOT_URLCONF = "notebook_urls"`, and Django only imports that module when the *first request* arrives — so we can build it now. Any module with a `urlpatterns` list qualifies… including one we assemble by hand with `types.ModuleType` and plant in `sys.modules`. (In the real project it's a `.py` file — which is just a module with a filename.)

Three views, ChurnScope-flavoured: a home page (`HttpResponse`), the predictions table (`render()` + a template we register under the same name the loader will look up — watch it query the **model** and fill the **template**: M, T and V in one function), and a JSON API (`JsonResponse`).

In [18]:
if HAS_DJANGO:
    from django.http import HttpResponse, JsonResponse
    from django.shortcuts import render

    TEMPLATE_REGISTRY["scoring/predictions.html"] = """\
<h1>Latest predictions</h1>
<table>
{% for p in predictions %}  <tr><td>{{ p.contract }}</td><td>{{ p.probability|floatformat:2 }}</td><td>{% if p.will_churn %}⚠️ churn{% else %}ok{% endif %}</td></tr>
{% endfor %}</table>"""

    def home(request):
        return HttpResponse("<h1>ChurnScope — notebook edition</h1>")

    def predictions_table(request):
        rows = Prediction.objects.order_by("-probability")[:5]
        return render(request, "scoring/predictions.html", {"predictions": rows})

    def api_predictions(request):
        rows = list(Prediction.objects.values("id", "contract", "probability", "will_churn")[:3])
        return JsonResponse({"count": Prediction.objects.count(), "results": rows})

    print("Three views defined — plain functions until a URL points at them.")
else:
    print("Django not installed — skipping.")

Three views defined — plain functions until a URL points at them.


In [19]:
if HAS_DJANGO:
    import types

    from django.urls import clear_url_caches, path

    urlconf = types.ModuleType("notebook_urls")        # ROOT_URLCONF, kept from §2
    urlconf.urlpatterns = [
        path("", home, name="home"),
        path("predictions/", predictions_table, name="predictions"),
        path("api/predictions/", api_predictions, name="api_predictions"),
    ]
    sys.modules["notebook_urls"] = urlconf
    clear_url_caches()    # Django caches the compiled resolver — flush after edits (🔬 below)

    print("Routing table:")
    for pattern in urlconf.urlpatterns:
        print(f"  /{pattern.pattern}  →  {pattern.callback.__name__}()")
else:
    print("Django not installed — skipping.")

Routing table:
  /  →  home()
  /predictions/  →  predictions_table()
  /api/predictions/  →  api_predictions()


Now the payoff. Django's test **`Client`** — the same one `manage.py test` and `scoring/tests.py` use — is a browser without the browser: it builds a real request, pushes it through *the entire pipeline* (URLconf → view → ORM → template), and hands back the real response object. No server process, no port, no HTTP over a wire:

In [20]:
if HAS_DJANGO:
    from django.test import Client

    client = Client()

    response = client.get("/")                 # request → URLconf → home() → response
    print("status      :", response.status_code)
    print("content-type:", response["Content-Type"])
    print("body        :", response.content.decode())
else:
    print("Django not installed — skipping.")

status      : 200
content-type: text/html; charset=utf-8
body        : <h1>ChurnScope — notebook edition</h1>


In [21]:
if HAS_DJANGO:
    print("GET /nope/ →", client.get("/nope/").status_code, "(no route matched)")
    # ↑ the red "Not Found: /nope/" line is Django's request log — runserver prints these too

    print(client.get("/predictions/").content.decode())       # HTML via model + template

    print("JSON:", client.get("/api/predictions/").json())     # dict, parsed for you

    from django.urls import reverse
    print("reverse('api_predictions') →", reverse("api_predictions"))
else:
    print("Django not installed — skipping.")

Not Found: /nope/


GET /nope/ → 404 (no route matched)
<h1>Latest predictions</h1>
<table>
  <tr><td>month-to-month</td><td>0.97</td><td>⚠️ churn</td></tr>
  <tr><td>month-to-month</td><td>0.92</td><td>⚠️ churn</td></tr>
  <tr><td>month-to-month</td><td>0.86</td><td>⚠️ churn</td></tr>
  <tr><td>one-year</td><td>0.52</td><td>⚠️ churn</td></tr>
  <tr><td>one-year</td><td>0.26</td><td>ok</td></tr>
</table>
JSON: {'count': 6, 'results': [{'id': 6, 'contract': 'two-year', 'probability': 0.0328, 'will_churn': False}, {'id': 5, 'contract': 'one-year', 'probability': 0.2621, 'will_churn': False}, {'id': 4, 'contract': 'one-year', 'probability': 0.515, 'will_churn': True}]}
reverse('api_predictions') → /api/predictions/


### 🔬 Named routes and the resolver cache

`name="api_predictions"` plus `reverse()` means templates, views and tests never hard-code paths — rename a URL once and everything follows. (The example app also namespaces with `app_name = "scoring"`, so there it's `reverse("scoring:api_score")`.) And about that `clear_url_caches()`: Django compiles `urlpatterns` into a resolver and caches it hard — perfect for a server, where routes never change at runtime; wrong for a notebook, where we keep editing the list. Forget to flush and your shiny new route 404s while you stare at a perfectly correct `urlpatterns`.

---

### ✋ Quick exercise (~2 min) — Call the API like a robot

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Use the test client to `GET /api/predictions/`. Assert the status code is `200` (exactly what a test would do), parse the body with `.json()`, and print the total row count plus the first result. Four lines, and you've written an integration test.

In [22]:
# ✍️ Your turn 👇
# response = client.get(...)
response = ...

# assert response.status_code == 200
# payload = response.json()
# print("rows in DB  :", payload["count"])
# print("first result:", payload["results"][0])

<details>
<summary>✅ <b>Solution</b></summary>

```python
if HAS_DJANGO:
    response = client.get("/api/predictions/")
    assert response.status_code == 200, f"expected 200, got {response.status_code}"
    payload = response.json()
    print("rows in DB  :", payload["count"])
    print("first result:", payload["results"][0])
else:
    print("Django not installed — skipping.")
```

That `assert` line is the heart of `example-app/scoring/tests.py` — the deployment chapter turns exactly this pattern into `self.assertEqual(resp.status_code, 200)` inside a `TestCase`. If you can call an endpoint with the client, you can test it in CI; no server, so the whole suite runs in milliseconds.
</details>

## 7. Forms — validation you don't write twice

Everything so far trusted its inputs. The web does not get that luxury: query strings and form posts arrive as **strings, typed by strangers**. A Django **form** declares the contract once — types, bounds, choices — and from that single definition you get validation, error messages, *and* HTML rendering. This is ChurnScope's `ChurnForm`, verbatim from `example-app/scoring/forms.py`:

In [23]:
if HAS_DJANGO:
    from django import forms

    CONTRACT_CHOICES = [
        ("month-to-month", "Month-to-month"),
        ("one-year", "One year"),
        ("two-year", "Two year"),
    ]

    class ChurnForm(forms.Form):
        """Validates one customer's details before they reach the scorer."""

        tenure_months = forms.IntegerField(min_value=0, max_value=120, initial=6)
        monthly_charges = forms.FloatField(min_value=0, initial=70.0)
        support_tickets = forms.IntegerField(min_value=0, max_value=50, initial=2)
        contract = forms.ChoiceField(choices=CONTRACT_CHOICES)

    # One definition → HTML for free (note min/max became widget attributes):
    print(ChurnForm()["tenure_months"])
    print(ChurnForm()["contract"])
else:
    print("Django not installed — skipping.")

<input type="number" name="tenure_months" value="6" min="0" max="120" required id="id_tenure_months">
<select name="contract" id="id_contract">
  <option value="month-to-month">Month-to-month</option>

  <option value="one-year">One year</option>

  <option value="two-year">Two year</option>

</select>


In [24]:
if HAS_DJANGO:
    raw_post = {"tenure_months": "3", "monthly_charges": "95",   # strings — like real POST data
                "support_tickets": "4", "contract": "month-to-month"}
    form = ChurnForm(raw_post)
    print("valid?       ", form.is_valid())
    print("cleaned_data :", form.cleaned_data)
    print("types        :", {k: type(v).__name__ for k, v in form.cleaned_data.items()})

    hostile = ChurnForm({"tenure_months": "-2", "monthly_charges": "lots", "contract": "weekly"})
    print("\nvalid?", hostile.is_valid(), "— per-field errors:")
    for field, errors in hostile.errors.items():
        print(f"  {field}: {list(errors)}")
else:
    print("Django not installed — skipping.")

valid?        True
cleaned_data : {'tenure_months': 3, 'monthly_charges': 95.0, 'support_tickets': 4, 'contract': 'month-to-month'}
types        : {'tenure_months': 'int', 'monthly_charges': 'float', 'support_tickets': 'int', 'contract': 'str'}

valid? False — per-field errors:
  tenure_months: ['Ensure this value is greater than or equal to 0.']
  monthly_charges: ['Enter a number.']
  support_tickets: ['This field is required.']
  contract: ['Select a valid choice. weekly is not one of the available choices.']


Strings went in; typed, range-checked Python came out — or a precise, per-field error report. Your view (and your model, and your scorer) never see a `"lots"` where a float belongs.

> ⚠️ **Pitfall: `cleaned_data` does not exist until you call `is_valid()`.** Validation *creates* it. Touch `form.cleaned_data` first and you get `AttributeError: 'ChurnForm' object has no attribute 'cleaned_data'` — the single most common Django beginner crash, and the star of Exercise 4. The discipline is always `if form.is_valid(): … form.cleaned_data …` — and never reach into `request.POST` yourself once you have a form.

---

### ✋ Quick exercise (~2 min) — Trust nothing from the outside

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

A payload arrives claiming `tenure_months="999"` and `contract="weekly"` (with `monthly_charges="80"`, `support_tickets="2"`). Bind it to a `ChurnForm`, show that it fails, and print which fields complain and why. Then fix the payload (tenure `99`, contract `two-year`) and print the typed `cleaned_data`.

In [25]:
# ✍️ Your turn 👇
suspicious = {"tenure_months": "999", "monthly_charges": "80",
              "support_tickets": "2", "contract": "weekly"}

# form = ChurnForm(...)
form = ...

# print("valid?", form.is_valid())
# for field, errors in form.errors.items():
#     print(f"  {field}: {list(errors)}")

# fixed = ChurnForm({...})    # tenure 99, contract "two-year"
# print("fixed →", fixed.is_valid(), fixed.cleaned_data)

<details>
<summary>✅ <b>Solution</b></summary>

```python
if HAS_DJANGO:
    suspicious = {"tenure_months": "999", "monthly_charges": "80",
                  "support_tickets": "2", "contract": "weekly"}
    form = ChurnForm(suspicious)
    print("valid?", form.is_valid())
    for field, errors in form.errors.items():
        print(f"  {field}: {list(errors)}")

    fixed = ChurnForm({"tenure_months": "99", "monthly_charges": "80",
                       "support_tickets": "2", "contract": "two-year"})
    print("fixed →", fixed.is_valid(), fixed.cleaned_data)
else:
    print("Django not installed — skipping.")
```

Two fields fail for two different reasons: `tenure_months` breaks a *bound* (`max_value=120`), `contract` isn't in the *choices* whitelist. Both rules live in one place — the form — and every consumer (HTML page, API, test) inherits them. Notice the fixed payload comes back as `{'tenure_months': 99, …}`: an `int`, not the `"99"` string that went in.
</details>

### The full pipeline — ChurnScope's form view, POSTed through the client

Time to connect every station. The **`score` view** below is `views.index` from the example app, line for line — renamed, and minus one cosmetic (the real view also hands its template the `threshold` for display): GET → show the empty form; POST → validate, call the scorer, **log the prediction via the ORM**, re-render with the result. One new template — `{{ form.as_p }}` renders every field with labels and errors in one tag, and `{% csrf_token %}` is Django's anti-forgery stamp, mandatory in any real POST form — plus a rebuilt routing table:

In [26]:
if HAS_DJANGO:
    TEMPLATE_REGISTRY["scoring/form.html"] = """\
<h1>ChurnScope — score a customer</h1>
<form method="post">
  {% csrf_token %}
  {{ form.as_p }}
  <button type="submit">Score</button>
</form>
{% if result %}<p id="result">Churn probability: <b>{{ result.probability|floatformat:2 }}</b> —
{% if result.will_churn %}likely to CHURN{% else %}likely to stay{% endif %}</p>{% endif %}"""

    THRESHOLD = 0.5

    def score(request):
        """GET shows the form; POST scores + logs + shows the result (= example-app index)."""
        result = None
        if request.method == "POST":
            form = ChurnForm(request.POST)
            if form.is_valid():
                data = form.cleaned_data
                p = churn_probability(**data)
                will_churn = p >= THRESHOLD
                Prediction.objects.create(probability=p, will_churn=will_churn, **data)
                result = {"probability": p, "will_churn": will_churn}
        else:
            form = ChurnForm()
        return render(request, "scoring/form.html", {"form": form, "result": result})

    urlconf.urlpatterns = [
        path("", home, name="home"),
        path("predictions/", predictions_table, name="predictions"),
        path("api/predictions/", api_predictions, name="api_predictions"),
        path("score/", score, name="score"),
    ]
    clear_url_caches()
    print("Routes:", ", ".join(f"/{p.pattern}" for p in urlconf.urlpatterns))
else:
    print("Django not installed — skipping.")

Routes: /, /predictions/, /api/predictions/, /score/


In [27]:
if HAS_DJANGO:
    page = client.get("/score/")               # GET → the empty form
    print("status:", page.status_code, "\n")
    print(page.content.decode()[:430], "…")
else:
    print("Django not installed — skipping.")

status: 200 

<h1>ChurnScope — score a customer</h1>
<form method="post">
  <input type="hidden" name="csrfmiddlewaretoken" value="tIFMZKVSe4voCTbEG4U14eAxe4WBozupBgGOZaEg0EsvAGNjtDIqNCNzSmPJIuml">
  <p>
    <label for="id_tenure_months">Tenure months:</label>
    <input type="number" name="tenure_months" value="6" min="0" max="120" required id="id_tenure_months">
    
    
  </p>

  
  <p>
    <label for="id_monthly_charges">Monthly charge …


In [28]:
if HAS_DJANGO:
    rows_before = Prediction.objects.count()

    response = client.post("/score/", {"tenure_months": 2, "monthly_charges": 99.5,
                                       "support_tickets": 4, "contract": "month-to-month"})
    print("status:", response.status_code)
    for line in response.content.decode().splitlines():
        if 'id="result"' in line or "likely to" in line:
            print(line.strip())

    print(f"rows before POST: {rows_before} → after: {Prediction.objects.count()}"
          "   ← the view logged it via the ORM")
else:
    print("Django not installed — skipping.")

status: 200
<p id="result">Churn probability: <b>0.96</b> —
likely to CHURN</p>
rows before POST: 6 → after: 7   ← the view logged it via the ORM


In [29]:
if HAS_DJANGO:
    import re

    rows_before = Prediction.objects.count()

    response = client.post("/score/", {"tenure_months": "-3", "monthly_charges": "free",
                                       "support_tickets": "4", "contract": "month-to-month"})
    print("status:", response.status_code, "(the form page re-renders — now with errors)")
    print("errors shown to the user:",
          re.findall(r"<li>([^<]+)</li>", response.content.decode()))
    print(f"rows: {rows_before} → {Prediction.objects.count()}   ← invalid input logs NOTHING")
else:
    print("Django not installed — skipping.")

status: 200 (the form page re-renders — now with errors)
errors shown to the user: ['Ensure this value is greater than or equal to 0.', 'Enter a number.']
rows: 7 → 7   ← invalid input logs NOTHING


## 8. The admin — the battery we *won't* fake in a notebook

One ChurnScope feature has no notebook equivalent: the **admin site**. It needs `django.contrib.admin` plus sessions, messages, static files, a staff user and a browser — it's an interactive web app in its own right. What's worth studying here is how *little* code buys it. This is all of `example-app/scoring/admin.py`:

```python
from django.contrib import admin

from .models import Prediction


@admin.register(Prediction)
class PredictionAdmin(admin.ModelAdmin):
    list_display = (
        "created_at",
        "contract",
        "tenure_months",
        "monthly_charges",
        "support_tickets",
        "probability",
        "will_churn",
    )
    list_filter = ("contract", "will_churn")
    # A Prediction is an audit log — nobody should edit one after the fact.
    # `_meta.fields` lists every model field, so this stays correct if fields change.
    readonly_fields = [f.name for f in Prediction._meta.fields]
```

Ten-ish declarative lines produce a searchable, filterable, paginated back-office over every prediction your `score` view logs — the "killer feature" of [models-and-admin.md](models-and-admin.md), and the reason a non-engineer can inspect an AI system's decisions without asking you for SQL. To see it live, run the example app with the 2-minute instructions in the [module README](README.md) (`migrate` → `runserver` → `createsuperuser`), then open `/admin/`.

## 🧪 Practice exercises

### Exercise 1 — ⭐ Which contract type churns hardest?

Produce a per-contract summary — **one query, no Python loop over rows**: the number of predictions and the average churn probability for each contract type, highest average first. (`values("contract")` + `annotate(...)` is the GROUP BY pattern from §4.) Bonus: add each group's maximum probability.

In [30]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
if HAS_DJANGO:
    per_contract = (
        Prediction.objects.values("contract")
        .annotate(n=Count("id"), avg_p=Avg("probability"), worst=Max("probability"))
        .order_by("-avg_p")
    )
    for row in per_contract:
        print(f"{row['contract']:<16} n={row['n']}   avg p={row['avg_p']:.3f}   max p={row['worst']:.3f}")
```

`values("contract")` sets the GROUP BY key; `annotate` then adds one aggregate column *per group* (contrast `aggregate`, which collapses the whole table into a single dict). The database does the grouping and hands Python three tiny rows — the same "push the loop into SQL" move as the N+1 fix. Month-to-month tops the board by a mile, which is exactly the coefficient the scorer gave it.
</details>

### Exercise 2 — ⭐⭐ A detail endpoint

Add `GET /api/predictions/<int:pk>/` returning **one** prediction as JSON (`id`, `contract`, `probability`, `will_churn`) — and `{"error": ...}` with **status 404** when no such row exists (catch `Prediction.DoesNotExist`). `<int:pk>` is a *path converter*: Django parses the integer out of the URL and passes it to the view as an argument. Remember the §6 drill — modify `urlconf.urlpatterns`, then `clear_url_caches()` — and test both the happy path and a missing id with the client.

In [31]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
if HAS_DJANGO:
    def api_prediction_detail(request, pk):          # pk arrives as a real int
        try:
            p = Prediction.objects.get(pk=pk)
        except Prediction.DoesNotExist:
            return JsonResponse({"error": f"no prediction with id {pk}"}, status=404)
        return JsonResponse({"id": p.pk, "contract": p.contract,
                             "probability": p.probability, "will_churn": p.will_churn})

    route = "api/predictions/<int:pk>/"
    urlconf.urlpatterns = [u for u in urlconf.urlpatterns if str(u.pattern) != route]
    urlconf.urlpatterns.append(path(route, api_prediction_detail, name="api_prediction_detail"))
    clear_url_caches()

    some_pk = Prediction.objects.order_by("pk").first().pk
    print(client.get(f"/api/predictions/{some_pk}/").json())
    print("missing id →", client.get("/api/predictions/999999/").status_code)
```

Two things to notice. First, the filter-then-append dance makes the cell idempotent — URL routing is *first match wins*, so appending a second `quick-fix` copy of a route would silently shadow nothing but clutter forever. Second, `get()` raising `DoesNotExist` is the ORM's way of saying "exactly-one failed"; converting that to a 404 JSON body is the standard API courtesy (the example app's `api_score` does the same for bad input with a 400).
</details>

### Exercise 3 — ⭐⭐ A history-table template

Register a template named `"scoring/history.html"` (drop it into `TEMPLATE_REGISTRY`) that renders the **five most recent** predictions as an HTML table: a counter column (`forloop.counter`), contract, probability to 2 decimals, and a **bold** "churn" / plain "stay" verdict — plus an `{% empty %}` row for a fresh database. Then load it *by name* with `django.template.loader.get_template` (the same loader machinery `render()` uses) and print the result. The example app's real history page (`scoring/templates/scoring/history.html`) is this table plus pagination — see the auth-and-history chapter.

In [32]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
if HAS_DJANGO:
    TEMPLATE_REGISTRY["scoring/history.html"] = """\
<h2>Prediction history</h2>
<table>
  <tr><th>#</th><th>contract</th><th>p</th><th>verdict</th></tr>
{% for p in rows %}  <tr><td>{{ forloop.counter }}</td><td>{{ p.contract }}</td><td>{{ p.probability|floatformat:2 }}</td><td>{% if p.will_churn %}<b>churn</b>{% else %}stay{% endif %}</td></tr>
{% empty %}  <tr><td colspan="4">No predictions logged yet.</td></tr>
{% endfor %}</table>"""

    from django.template.loader import get_template

    tpl = get_template("scoring/history.html")       # found via the locmem loader
    print(tpl.render({"rows": Prediction.objects.all()[:5]}))
```

`Prediction.objects.all()[:5]` needs no `order_by`: the model's `Meta.ordering = ["-created_at"]` already means *newest first*, everywhere — one rule, defined once, inherited by the admin, this table, and the real history page alike. `get_template` walks the configured loaders exactly like `render()` does; registering the string under a name is all it took to be findable.
</details>

### Exercise 4 — ⭐⭐ Debug me 🐞

A colleague wants scoring over a plain GET endpoint and wrote the view below. It uses `RequestFactory` — the test client's little sibling: it *builds* a request object without routing it, so you can call any view as a plain function. The cell crashes. Run it, read the traceback bottom-up, find the missing line, and fix the view so valid input returns JSON and invalid input returns a **400** with the errors.

In [33]:
# ⚠️ THIS CELL INTENTIONALLY ERRORS — that's the exercise. Fix the view!
if HAS_DJANGO:
    from django.test import RequestFactory

    def quick_score(request):
        form = ChurnForm(request.GET)
        data = form.cleaned_data                       # 💥 crashes here — why?
        p = churn_probability(**data)
        return JsonResponse({"probability": round(p, 4)})

    request = RequestFactory().get("/quick/", {"tenure_months": "2", "monthly_charges": "99",
                                               "support_tickets": "4", "contract": "month-to-month"})
    print(quick_score(request).content.decode())
else:
    print("Django not installed — skipping.")

AttributeError: 'ChurnForm' object has no attribute 'cleaned_data'

<details>
<summary>💡 <b>Solution</b></summary>

```python
if HAS_DJANGO:
    from django.test import RequestFactory

    def quick_score(request):
        form = ChurnForm(request.GET)
        if not form.is_valid():                        # ← the missing line
            return JsonResponse({"errors": dict(form.errors)}, status=400)
        p = churn_probability(**form.cleaned_data)
        return JsonResponse({"probability": round(p, 4)})

    factory = RequestFactory()
    ok = quick_score(factory.get("/quick/", {"tenure_months": "2", "monthly_charges": "99",
                                             "support_tickets": "4", "contract": "month-to-month"}))
    print(ok.status_code, ok.content.decode())

    bad = quick_score(factory.get("/quick/", {"tenure_months": "banana"}))
    print(bad.status_code, bad.content.decode())
```

The crash was `AttributeError: 'ChurnForm' object has no attribute 'cleaned_data'` — §7's pitfall in the wild. **`is_valid()` creates `cleaned_data`**; skip validation and the attribute simply isn't there. The fix is the standard two-step: validate first, and on failure return `form.errors` (a perfectly JSON-serialisable dict) with status 400 — the same contract the example app's `api_score` view keeps with its try/except.
</details>

## 🧠 Stretch exercises

### Stretch exercise A — ⭐⭐⭐ A paginated history page

Recreate the example app's `/history/` page (minus the login — that's Lab 2's business): a view that wraps `Prediction.objects.all()` in a `django.core.paginator.Paginator` (3 per page), reads `?page=` from `request.GET`, and renders a registered template showing "page X of Y", the rows, and a *next* link only when `page_obj.has_next`. Route it at `history/` and fetch pages 1 and 2 with the client — then request `?page=banana` and admire `get_page()`'s clamping (no crash: nonsense → page 1).

In [34]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
if HAS_DJANGO:
    from django.core.paginator import Paginator

    TEMPLATE_REGISTRY["scoring/paged_history.html"] = (
        "<h2>History — page {{ page_obj.number }} of {{ page_obj.paginator.num_pages }}</h2>\n"
        "{% for p in page_obj %}<p>{{ p.contract }} — {{ p.probability|floatformat:2 }}</p>\n{% endfor %}"
        "{% if page_obj.has_next %}<a href=\"?page={{ page_obj.next_page_number }}\">next</a>{% endif %}")

    def history(request):
        paginator = Paginator(Prediction.objects.all(), per_page=3)
        page_obj = paginator.get_page(request.GET.get("page"))
        return render(request, "scoring/paged_history.html", {"page_obj": page_obj})

    urlconf.urlpatterns = [u for u in urlconf.urlpatterns if str(u.pattern) != "history/"]
    urlconf.urlpatterns.append(path("history/", history, name="history"))
    clear_url_caches()

    print(client.get("/history/").content.decode())
    print(client.get("/history/?page=2").content.decode())
    print("?page=banana →", client.get("/history/?page=banana").status_code, "(clamped, not crashed)")
```

Except for the template name and the page size (the real app uses `per_page=10`; 3 makes our small table actually paginate), `history` is the example app's view (before `@login_required` lands on it in the auth chapter). Two batteries doing quiet work: the paginator slices the *lazy* queryset, so the database only ever returns three rows per request (`LIMIT 3 OFFSET …`), and `get_page()` swallows garbage input instead of 500ing. Ordering, as ever, comes free from `Meta.ordering`.
</details>

### Stretch exercise B — ⭐⭐⭐ Re-threshold the whole table in two queries

The business moves the churn threshold from 0.5 to **0.6**. Recompute `will_churn` for *every* logged prediction in **exactly two SQL UPDATEs** — no Python loop, no per-row `save()` (`reset_queries()` + `connection.queries` is your referee). Then put the threshold back to 0.5 the same way, so the rest of the notebook sees consistent data.

In [35]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
if HAS_DJANGO:
    reset_queries()
    up = Prediction.objects.filter(probability__gte=0.6).update(will_churn=True)
    down = Prediction.objects.filter(probability__lt=0.6).update(will_churn=False)
    print(f"threshold 0.6 → {up} churn / {down} stay, in {len(connection.queries)} queries")

    # restore the 0.5 threshold (same two-query pattern, so the notebook stays consistent)
    Prediction.objects.filter(probability__gte=0.5).update(will_churn=True)
    Prediction.objects.filter(probability__lt=0.5).update(will_churn=False)
    print("churners at 0.5 again:", Prediction.objects.filter(will_churn=True).count())
```

`queryset.update()` compiles to a single `UPDATE … SET … WHERE …` — the loop happens inside the database, exactly like the N+1 fix. The per-row alternative (`for p in …: p.will_churn = …; p.save()`) issues one UPDATE *per row* and also re-writes every column each time. One caveat worth knowing: `update()` bypasses `save()` and signals, and for arithmetic on existing values (`price = price * 1.1`) you'd reach for `F()` expressions to stay race-free.
</details>

### Stretch exercise C — ⭐⭐⭐ A CSV export view

Every business tool eventually grows an "Export to Excel" button. Write `export_csv(request)` that returns an `HttpResponse` with `content_type="text/csv"` and a `Content-Disposition: attachment; filename="predictions.csv"` header, its body written by `csv.writer` *directly onto the response* (a response object is file-like). One header row, then `contract, tenure_months, probability, will_churn` for every prediction — `values_list` hands you ready-made tuples. Route it at `export.csv` and print the first few lines through the client.

In [36]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
if HAS_DJANGO:
    import csv

    def export_csv(request):
        response = HttpResponse(content_type="text/csv")
        response["Content-Disposition"] = 'attachment; filename="predictions.csv"'
        writer = csv.writer(response)                  # a response is file-like!
        writer.writerow(["contract", "tenure_months", "probability", "will_churn"])
        for row in Prediction.objects.values_list("contract", "tenure_months",
                                                  "probability", "will_churn"):
            writer.writerow(row)
        return response

    urlconf.urlpatterns = [u for u in urlconf.urlpatterns if str(u.pattern) != "export.csv"]
    urlconf.urlpatterns.append(path("export.csv", export_csv, name="export_csv"))
    clear_url_caches()

    response = client.get("/export.csv")
    print(response.status_code, response["Content-Type"], "|", response["Content-Disposition"])
    print("\n".join(response.content.decode().splitlines()[:4]))
```

A view doesn't have to return HTML or JSON — a response is just headers plus a body, and `csv.writer` happily treats `HttpResponse` as its file. The `Content-Disposition` header is what turns "display this" into "download this as *predictions.csv*". For millions of rows you'd switch to `StreamingHttpResponse`; the shape of the view stays the same.
</details>

## 🎁 Bonus mini-project — The `/api/stats/` endpoint

ChurnScope's product manager wants one dashboard number per contract type: *"How is churn risk distributed across our contracts?"* Build `GET /api/stats/` returning JSON of this shape (the *shape* is the contract — your exact numbers depend on which checkpoint and exercise cells you've run):

```json
{"total": 8,
 "per_contract": [
   {"contract": "month-to-month", "n": 5, "avg_probability": 0.94, "churn_share": 1.0},
   {"contract": "one-year",       "n": 2, "avg_probability": 0.39, "churn_share": 0.5},
   {"contract": "two-year",       "n": 1, "avg_probability": 0.03, "churn_share": 0.0}]}
```

- one `values("contract").annotate(...)` queryset with `Count` and `Avg` (§4 + Exercise 1),
- `churn_share` = the share of rows with `will_churn=True` — hint: `Avg("will_churn", output_field=FloatField())` works because booleans are stored as 0/1,
- ordered riskiest-first, wired into the URLconf, verified with the client.

This is the shape of every real "stats endpoint" you'll ever ship: one annotated queryset, one `JsonResponse`.

In [37]:
# Your code here  👇


<details>
<summary>💡 <b>Solution sketch</b></summary>

```python
if HAS_DJANGO:
    from django.db.models import FloatField

    def api_stats(request):
        per_contract = list(
            Prediction.objects.values("contract")
            .annotate(n=Count("id"),
                      avg_probability=Avg("probability"),
                      churn_share=Avg("will_churn", output_field=FloatField()))
            .order_by("-avg_probability")
        )
        return JsonResponse({"total": Prediction.objects.count(),
                             "per_contract": per_contract})

    urlconf.urlpatterns = [u for u in urlconf.urlpatterns if str(u.pattern) != "api/stats/"]
    urlconf.urlpatterns.append(path("api/stats/", api_stats, name="api_stats"))
    clear_url_caches()

    import json
    print(json.dumps(client.get("/api/stats/").json(), indent=2))
```

The whole endpoint is ~10 lines because each battery does its part: the ORM groups and averages inside SQLite, `JsonResponse` serialises the list of dicts, the URLconf routes it, and the client proves it works. The `output_field=FloatField()` nudge matters: without it Django faithfully converts the average of a `BooleanField` back to a boolean, and your PM's dashboard would read `true` where it should read `1.0`. Want it on a real dashboard? This JSON is one `requests.get` away from the pandas/matplotlib pipeline of Module 2.
</details>

## 🧠 Key takeaways

> 🧭 **The story in one line.** We rebuilt ChurnScope's engine room inside one kernel — settings by hand, a real SQLite database, the very same `Prediction` model, templates, routed views, validated forms — and drove requests through the full pipeline: **request → URLconf → view → model/ORM → template → response.**

1. **Django is configuration plus conventions.** `settings.configure()` + `django.setup()` is everything `manage.py` bootstraps; an app is just an importable package; every management command is also `call_command(...)`.
2. **Models are classes, rows are instances**, `objects` is the gateway — and the `CREATE TABLE` is *derived* from the class. Migrations version that schema like Git versions code.
3. **QuerySets are lazy** and compile to parameterised SQL you can read (`str(qs.query)`); with `DEBUG=True`, `connection.queries` shows what actually ran.
4. **Push loops into the database** — `annotate`/`aggregate`, `queryset.update()`, pagination — or meet the N+1 problem in production.
5. **Templates are dumb on purpose**: variables, tags, filters, nothing else. Logic lives in views; formatting lives in templates.
6. **A view is request → response**; the URLconf is a list in a module; the test **`Client`** exercises the whole pipeline with no server — which is exactly how Django apps are tested in CI.
7. **Forms turn strangers' strings into typed, bounded data.** `is_valid()` first, `cleaned_data` after — never the other way (Exercise 4 has the scar).
8. **The admin is ~10 declarative lines** for a full back-office — the one battery a notebook can't render; the example app can.

## ✅ Self-assessment

- [ ] I can boot Django in a plain Python process and explain what `settings.configure` + `django.setup` do
- [ ] I can define a model, get its table created, and say what migrations add over `run_syncdb`
- [ ] I can write `filter` / `exclude` / `order_by` / lookup queries and predict *when* the SQL actually runs
- [ ] I can group-and-aggregate in one query instead of looping row by row — and prove it with `connection.queries`
- [ ] I can render a template with loops, conditionals and filters
- [ ] I can add a URL → view pair and verify it end-to-end with the test client
- [ ] I can explain why `cleaned_data` needs `is_valid()` first, and return a 400 with `form.errors`

## 🚀 Next step

Two short roads from here:

- **Run the real thing.** [`example-app/`](example-app/) is this lab with files instead of cells — the [module README](README.md)'s 2-minute instructions (`migrate` → `runserver`) put ChurnScope in your browser. Then `createsuperuser`, open `/admin/`, and read [models-and-admin.md](models-and-admin.md) beside it — the admin is the battery this lab could only describe.
- **Lab 2 — `lab02_serving_a_model_with_auth.ipynb`** picks up the remaining chapters: serving a *real* trained model behind the view ([serving-a-model.md](serving-a-model.md)), plus `@login_required`, users, and the protected history page ([auth-and-history.md](auth-and-history.md)).

> 🚀 Open a notebook. Edit one number. Re-run. *That* is the whole craft.